# Historical experiment notebook
This notebook preserves its original protocol and embedded source. For offline result reproduction use `scripts/reproduce.py` in the repository. For a new measurement keep the original experiment identity and do not overwrite old results. Archive checks are intentionally strict.


# CPU–GPU Research — Step 3 of 5
## Freeze predictions, then measure new workloads

**Use a fresh Colab T4 GPU runtime before running any cells.** The original inference engine and Step 1/2 code are unchanged.
Upload only your `step2_20260919T194826078232Z_export.zip` when prompted.

This step is a **new hardware experiment**, unlike the CPU-only fitting steps. Four previously unmeasured batch/prompt combinations are tested at five placements, plus three historical controls. Two order-reversed blocks yield **230 measured generations** (200 primary + 30 control), with warmups excluded.

No predictor is retrained and the memory margin stays at 5% + 16 MiB. Predictions and policy decisions are saved before any new measurements. The budgets constrain peak PyTorch allocation conceptually; they are not physical GPU limits.

**Roadmap:** 1 latency model ✓ → 2 memory/planner ✓ → **3 new-workload test** → 4 focused robustness + interpretation → 5 report/GitHub/academic CV.

Run Sections 1–8 in order. Do not run another benchmark simultaneously. **On failure, stop and run Section 8 to preserve the partial export; do not repeatedly rerun a slow or failed case.**

The local CPU test report has 185 passing tests and nine CUDA skips. The full T4 and pretrained-reference experiment has not been run by the assistant.

## 1. Install the self-contained source

Keep Colab’s CUDA-enabled PyTorch. This cell installs the other dependencies without replacing PyTorch.
The embedded source is verified before extraction. It contains no model weights or benchmark outcomes.

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

SOURCE_ARCHIVE_SHA256 = "e3ec708bf396eb4f400c2f938ed1b55aa6a144a1125dedc9a094584a210a315b"
SOURCE_ARCHIVE_B64 = "..."

WORK = Path(os.environ.get("OFFLOAD_STEP3_WORK", str(Path.cwd() / "cpu_gpu_research_step3"))).resolve()
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK / "source"
SOURCE.mkdir(exist_ok=True)
blob = base64.b64decode(SOURCE_ARCHIVE_B64)
assert hashlib.sha256(blob).hexdigest() == SOURCE_ARCHIVE_SHA256, "Embedded source hash mismatch"
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for info in z.infolist():
        target = (SOURCE / info.filename).resolve()
        if not target.is_relative_to(SOURCE.resolve()):
            raise RuntimeError("Unsafe source archive member")
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        content = z.read(info.filename)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError("Existing source was modified: " + str(target))
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
(SOURCE / "step3_source.zip").write_bytes(blob)
os.chdir(SOURCE)
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
# Local verification can skip dependency installation; normal Colab runs install it.
if os.environ.get("OFFLOAD_STEP3_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-step3.txt"], check=True)
print("Source ready:", SOURCE)
print("PyTorch was not replaced by this cell.")


## 2. Upload your Step 2 checkpoint

Select **only** `step2_20260919T194826078232Z_export.zip`. It already contains the original baseline.
The input hash is checked in Section 4. Uploaded code is not executed; the notebook uses the embedded verified source.

In [ ]:
LOCAL_STEP2_ZIP = ""  # Optional: full path when running outside Colab.
local = os.environ.get("OFFLOAD_STEP2_EXPORT", LOCAL_STEP2_ZIP)
if local:
    STEP2 = Path(local).expanduser().resolve()
    if not STEP2.is_file():
        raise FileNotFoundError(STEP2)
else:
    from google.colab import files
    uploaded = files.upload()
    names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Upload exactly one Step 2 export ZIP.")
    STEP2 = (SOURCE / names[0]).resolve()
    STEP2.write_bytes(uploaded[names[0]])
    del uploaded

if "OUT" not in globals():
    OUT = WORK / "results" / ("step3_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    OUT.mkdir(parents=True, exist_ok=False)
shutil.copyfile(SOURCE / "step3_source.zip", OUT / "step3_source.zip")
for name in ("README_STEP3.md", "TEST_STATUS_STEP3.md"):
    shutil.copyfile(SOURCE / name, OUT / name)
print("Checkpoint:", STEP2.name)
print("Output:", OUT)

def run_logged(arguments, filename):
    """Stream logs and stop the child process on a notebook interruption."""
    log = OUT / filename
    if log.exists():
        log = log.with_name(log.stem + "_" + datetime.now(timezone.utc).strftime("%H%M%S%f") + log.suffix)
    with log.open("w", encoding="utf-8") as target:
        process = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, cwd=SOURCE)
        try:
            for line in process.stdout:
                print(line, end="")
                target.write(line)
                target.flush()
            result = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            if process.stdout is not None:
                process.stdout.close()
    if result != 0:
        raise subprocess.CalledProcessError(result, arguments)
    return log

def save_checkpoint():
    result = subprocess.check_output([sys.executable, "-m", "offload_research.fresh", "export", "--out", str(OUT)],
                                     cwd=SOURCE, text=True)
    path = Path(json.loads(result))
    print("Checkpoint saved:", path)
    return path


## 3. Run software tests

All tests should pass in a T4 runtime. The local CPU check had **185 passed / 9 CUDA skipped**; those nine CUDA tests should execute here. The new reference gate comes later and is distinct from these software tests.

In [ ]:
run_logged([sys.executable, "-m", "pytest", "-q"], "test_output.txt")
(OUT / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")


## 4. Freeze models, workloads, predictions, and decisions

The two latency models and memory model are loaded unchanged from Step 2. The old raw data is audited, not refitted.
This writes **23 candidate predictions** (20 primary + 3 controls), **96 primary policy/budget decisions**, and a checksum seal.

These values are **predictions, not new benchmark results**. Do not alter the margin or protocol after observing measurements.
A local seal records integrity; it is not a public preregistration.

In [ ]:
command = "verify" if (OUT / "frozen" / "seal.json").exists() else "prepare"
arguments = [sys.executable, "-m", "offload_research.fresh", command, "--out", str(OUT)]
if command == "prepare":
    arguments += ["--step2", str(STEP2)]
run_logged(arguments, "freeze_log.txt")
import pandas as pd
protocol = json.loads((OUT / "frozen" / "protocol.json").read_text())
print("Primary settings:", protocol["primary_candidates"])
print("Historical controls:", protocol["historical_controls"])
print("Planned measured generations:", protocol["expected_generation_trials"])
predictions = pd.read_csv(OUT / "frozen" / "predictions.csv")
display(predictions.loc[(predictions.kind == "primary") & (predictions.gpu_layers == 6),
    ["workload_id", "gpu_layers", "predicted_peak_mib", "guarded_peak_mib", "predicted_generation_ms_compute_transfer"]])
PREMEASUREMENT_ARCHIVE = Path(shutil.make_archive(str(OUT) + "_PREMEASUREMENT_ONLY", "zip", root_dir=OUT))
print("Pre-measurement archive:", PREMEASUREMENT_ARCHIVE)
print("No new GPU inference has been collected yet.")


## 5. Check the T4 runtime and pretrained numerical correctness

The hardware gate refuses other GPUs and records CPU/software differences from the baseline.
Then an isolated process performs **20 Hugging Face comparisons**: all four new prompt shapes × five placements.
Each checks final-position prefill logits and **three teacher-forced cached steps**. It is not task accuracy or verification of every output token.

The reference process exits before measurements, preventing its weights and caches from contaminating GPU memory results.
Do not bypass a failing gate. On error, use Section 8 to save the partial archive.

In [ ]:
try:
    run_logged([sys.executable, "-m", "offload_research.collect_fresh", "preflight", "--out", str(OUT)], "hardware_log.txt")
    if (OUT / "reference_validation.json").exists():
        gate = json.loads((OUT / "reference_validation.json").read_text())
        if gate.get("status") != "passed":
            raise RuntimeError("An earlier reference gate failed. Preserve it; do not overwrite.")
        print("Existing reference result retained; its identity is checked again by the collector.")
    else:
        run_logged([sys.executable, "-m", "offload_research.collect_fresh", "reference", "--out", str(OUT)], "reference_log.txt")
finally:
    CHECKPOINT_ARCHIVE = save_checkpoint()


## 6. Collect the two fixed measurement blocks

There are **23 settings per block**, **five warmups**, and **five timed repetitions**. Block 2 reverses block 1's order.
The result is **230 timed generations**, not a new 45-case full sweep. File writes and validation are outside the timing intervals.

Do not change threads, code, precision, candidates, or budgets. Do not run another notebook's benchmark at the same time.
A started but incomplete block is not silently rerun. Its completed trials remain in the export. Both blocks must use the same runtime.

In [ ]:
try:
    for block in (0, 1):
        block_dir = OUT / "measurement" / f"block_{block}"
        if (block_dir / "completed.json").exists():
            print("Retaining completed block", block + 1)
            continue
        if block_dir.exists():
            raise RuntimeError("An incomplete block already exists. Export this attempt rather than overwriting it.")
        run_logged([sys.executable, "-m", "offload_research.collect_fresh", "collect", "--out", str(OUT),
                    "--block", str(block)], f"measurement_block_{block}.log")
finally:
    CHECKPOINT_ARCHIVE = save_checkpoint()


## 7. Score frozen predictions against the new measurements

This cell does not retrain. It computes latency/memory prediction errors, actual budget violations, and placement regret against the same five measured candidates. Historical controls are reported separately and never used to adjust predictions.

If any candidate is missing or failed, the export is marked incomplete rather than claiming a complete oracle comparison.
Two blocks in one session and ten generations per setting do not establish production tail latency or cross-session confidence.

In [ ]:
run_logged([sys.executable, "-m", "offload_research.fresh", "analyze", "--out", str(OUT)], "analysis_log.txt")
coverage = json.loads((OUT / "analysis" / "coverage.json").read_text())
if coverage["complete"]:
    print("Fresh measurements — not retrospective cross-validation:")
    display(pd.read_csv(OUT / "analysis" / "prediction_metrics.csv"))
    display(pd.read_csv(OUT / "analysis" / "policy_metrics.csv"))
    display(pd.read_csv(OUT / "analysis" / "historical_controls.csv")[[
        "gpu_layers", "generation_ms_median", "generation_ms_median_historical", "generation_ratio_vs_historical"]])
else:
    print("Incomplete experiment. Preserve all failures and proceed to export.")
    display(pd.DataFrame(coverage["failures"]))


## 8. Download the checkpoint and stop

Run this even if a hardware/reference/measurement cell failed, as long as `OUT` exists. The export marks incomplete data explicitly.
Share the **`step3_..._export.zip`**, not the `PREMEASUREMENT_ONLY` ZIP.

Do not tune the model or margin after this first look. Step 4 will decide the focused repeat/robustness analysis; Step 5 is the report and public artifact.

In [ ]:
CHECKPOINT_ARCHIVE = save_checkpoint()
print("Share this file:", CHECKPOINT_ARCHIVE.name)
try:
    from google.colab import files
    files.download(str(CHECKPOINT_ARCHIVE))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(CHECKPOINT_ARCHIVE)))


### Technical references and scope

[PyTorch benchmarking](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html) explains warmup, synchronization, thread settings, and runtime variability.
[PyTorch peak allocation](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.memory.max_memory_allocated.html) is not total device usage.
[Colab availability](https://research.google.com/colaboratory/faq.html) does not guarantee a particular accelerator.

Read `README_STEP3.md` for the protocol, source provenance, failure handling, and interpretation limits.
No benchmark speedup, prediction accuracy, or policy improvement has been prefilled in this notebook.